# Rescaling with $\kappa_b$: Floquet exponent of the unstable pair (20/10/2026)

Report v2, point 10: can curves obtained at different $\kappa_b$ be rescaled onto each other?
This notebook only computes the raw curves; no rescaling yet.

**Quantity.** The period-2 satellite (the orbit of `Hill_method_doubling_corrected.ipynb`, $\varepsilon_d = 10.12\,g_2$) loses stability through a
Neimark–Sacker bifurcation: one complex pair of Floquet multipliers $\mu_{NS}$ crosses $|\mu| = 1$ at $\arg\mu\simeq\pm0.2\pi$.
We follow that pair and plot its Floquet exponent
$$\Lambda = \frac{\ln|\mu_{NS}|}{T},\qquad T = \frac{4\pi}{\omega_a}\ \text{(period of the period-2 orbit)},$$
against $g_2$, for several $\kappa_b$. $\Lambda < 0$ is the decay rate of perturbations along that pair, $\Lambda > 0$ their growth rate.

**Conventions.**
- $g_2 = E_J\sin(\varepsilon_p)\varphi_a^2\varphi_b$.
- $\kappa_b$ is quoted in the GKSL (article) convention, $\kappa_b/2\pi = 48$–$384$ MHz. The code damping is $-2\kappa\dot\phi_b$ with $\kappa = \kappa_b/2$ (96 MHz $\leftrightarrow$ $\kappa = \pi/10.4$).
- Small $\kappa_b$ are left out: there the threshold is too close to the conservative collision ($g_{2,0}\simeq76.2$ MHz) to resolve.

**Method.** Harmonic balance `catchaos.ats.HB2` (as in `Hill_method_cat_fold.ipynb`): basis over two drive periods, $\tau_d = \omega_a t/2$,
$M = \mathrm{diag}(1/4, 1/16)$, $C = \mathrm{diag}(0, \kappa/2\omega_b)$, drive on harmonic 4, exact one-period AFT.
Floquet exponents from Hill's method with the genuine-exponent selection of the previous notebooks (`catchaos.hill.hill_exponents`). Liouville check: the four exponents sum to $-\kappa_b$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, "/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/pylib")
from catchaos.params import w_a, w_b, phi_a, phi_b, E_J, MHz   # not kappa_b: here kappa_b is in the GKSL convention
from catchaos.plotting import *                                # width_in, height_in, ...
from catchaos.ats import HB2
from catchaos.hill import odd_content, hill_exponents
from catchaos.seeds import period2_seed

In [ ]:
# circuit: catchaos.params. g2_of kept here (its operation order; catchaos.params.g2_of = sin(eps_p) G0 differs in the last bit)
def g2_of(eps_p):
    return np.sin(eps_p)*E_J*phi_a**2*phi_b

ratio = 10.12                          # eps_d / g2
T = 4*np.pi/w_a                        # period of the period-2 orbit (ns)
kappa_b_ref = 2*np.pi/10.4             # GKSL kappa_b of the seed orbit (96 MHz)
kappa_b_list = 2*np.pi*np.linspace(0.048,0.384,10)#[0.048, 0.096, 0.192, 0.384])   # GKSL kappa_b (1/ns)
colors = plt.cm.viridis(np.linspace(0, 0.9, len(kappa_b_list)))   # one color per kappa_b

## Harmonic balance over two drive periods
`catchaos.ats.HB2`: $z = [\mathbf c_0, \mathbf s_1, \mathbf c_1, \dots]$, each block $(a, b)$, $\mathbf X = \mathbf c_0/\sqrt2 + \sum_k \mathbf s_k\sin k\tau_d + \mathbf c_k\cos k\tau_d$.
Its damping is the code $\kappa = \kappa_b/2$ (GKSL $\kappa_b$): $C = \mathrm{diag}(0, \kappa/\omega_b)$ in $\tau$, $\mathrm{diag}(0, \kappa/2\omega_b)$ in $\tau_d$.

In [ ]:
def set_params(eps_p, eps_d, kappa_b):
    # GKSL kappa_b -> code kappa = kappa_b/2
    hb.set_kappa(kappa_b/2)
    hb.set_params(eps_p, eps_d)

## Floquet exponents (Hill's method)
`catchaos.hill.hill_exponents` keeps the $2n = 4$ genuine exponents among the Hill eigenvalues (`select_genuine`, as in `Hill_method_cat_fold.ipynb`), in units of $1/\tau_d$.
`floquet_exponents` returns them in 1/ns, so that $\mu = e^{\lambda T}$.

In [ ]:
def floquet_exponents(hb, z):
    # genuine exponents of the Hill quadratic eigenproblem in tau_d, converted to 1/ns
    return hill_exponents(hb, z)*w_a/2

## Sweep
1. The seed (period-2 orbit at $\varepsilon_p = 1.2$, $\kappa_b/2\pi = 96$ MHz, converged with the former $J_1$ definition of $g_2$) is continued in $\varepsilon_d$ up to $10.12\,g_2$,
   then in $\kappa_b$ at $\varepsilon_p = 1.2$.
2. For each $\kappa_b$, $\varepsilon_p$ is decreased in steps of `d_eps_p` until the period-2 branch ends: it merges with the period-1 satellite
   (odd harmonics below `odd_min`, period doubling) or the solver fails (fold at large $\kappa_b$).
3. The four multipliers are stored at every point.

In [ ]:
N_H = 15
hb = HB2(N_H, kappa=kappa_b_ref/2)
eps_p_top, d_eps_p, eps_p_bottom = 1.2, 0.0005, 0.5   # fine step: Lambda is divided by kappa_b^2 later
odd_min = 1e-5

# period-2 orbit at (eps_p = 1.2, kappa_b/2pi = 96 MHz), J1 definition (catchaos.seeds.Z_P2_J1), continued in eps_d up to 10.12 g2
z_ref = period2_seed(hb, eps_p_top, ratio)

def continue_in_kappa(z, kappa_from, kappa_to, n_steps=20):
    for kap in np.linspace(kappa_from, kappa_to, n_steps + 1)[1:]:
        set_params(eps_p_top, ratio*g2_of(eps_p_top), kap)
        z = hb.solve(z)
    return z

In [ ]:
sweeps = {}       # k -> (eps_p, mu (n_points, 4))
for k, kappa_b in enumerate(kappa_b_list):
    z = continue_in_kappa(z_ref, kappa_b_ref, kappa_b)
    eps_k, mu_k, end = [], [], f"reached eps_p = {eps_p_bottom}"
    for e in np.arange(eps_p_top, eps_p_bottom - 1e-12, -d_eps_p):
        set_params(e, ratio*g2_of(e), kappa_b)
        try:
            z = hb.solve(z)
        except RuntimeError:
            end = f"solver failed at g2/2pi = {MHz(g2_of(e)):.2f} MHz (fold)"
            break
        if odd_content(z, N_H) < odd_min:
            end = f"merged with the period-1 satellite at g2/2pi = {MHz(g2_of(e)):.2f} MHz (period doubling)"
            break
        lam = floquet_exponents(hb, z)
        eps_k.append(e)
        mu_k.append(np.exp(lam*T))
    sweeps[k] = (np.array(eps_k), np.array(mu_k))
    liouville = np.max(np.abs(np.sum(np.log(np.abs(sweeps[k][1])), axis=1)/T + kappa_b))
    print(f"kappa_b/2pi = {MHz(kappa_b):3.0f} MHz: {len(eps_k)} points, {end}. Liouville residual {liouville:.1e} /ns")

## The Neimark–Sacker pair
At the top of the sweep ($g_2/2\pi = 128$ MHz) the unstable pair is the complex pair with the largest $|\mu|$. It is then followed down in $g_2$
by continuity: at each step, the multiplier (upper half-plane) closest to the previous one. $\Lambda = \ln|\mu_{NS}|/T$.
The threshold $g_2^*$ is the zero of $\Lambda$ (linear interpolation).

In [ ]:
def track_ns_pair(mu):
    # mu: (n_points, 4), ordered from the top of the sweep down. Returns mu_NS (upper half-plane member) at each point.
    up = [m[m.imag >= -1e-12] for m in mu]
    cplx = up[0][np.abs(up[0].imag) > 1e-9]
    out = [cplx[np.argmax(np.abs(cplx))]]
    for m in up[1:]:
        out.append(m[np.argmin(np.abs(m - out[-1]))])
    return np.array(out)

Lambda, g2_MHz, arg_ns, g2_star = {}, {}, {}, {}
for k, kappa_b in enumerate(kappa_b_list):
    eps_k, mu_k = sweeps[k]
    mu_ns = track_ns_pair(mu_k)
    g2_MHz[k] = MHz(g2_of(eps_k))
    Lambda[k] = np.log(np.abs(mu_ns))/T
    arg_ns[k] = np.angle(mu_ns)/np.pi
    i = np.where(np.diff(np.sign(Lambda[k])) != 0)[0]
    if len(i):
        i = i[-1]       # lowest crossing
        g2_star[k] = np.interp(0, Lambda[k][[i + 1, i]], g2_MHz[k][[i + 1, i]])
        print(f"kappa_b/2pi = {MHz(kappa_b):3.0f} MHz: g2*/2pi = {g2_star[k]:.2f} MHz, arg mu_NS = {np.interp(g2_star[k], g2_MHz[k][::-1], arg_ns[k][::-1]):.3f} pi, "
              f"branch ends at {g2_MHz[k][-1]:.2f} MHz with Lambda = {Lambda[k][-1]:.3f} /ns")
    else:
        print(f"kappa_b/2pi = {MHz(kappa_b):3.0f} MHz: no crossing on the branch")

In [ ]:
fig, ax = plt.subplots()
for k, kappa_b in enumerate(kappa_b_list):
    ax.plot(g2_MHz[k], Lambda[k], color=colors[k], label=rf"$\kappa_b/2\pi = {MHz(kappa_b):.0f}$ MHz")
ax.axhline(0, ls="--", color="grey", lw=0.8)
ax.set_xlim(70, 110)
ax.grid(False)
ax.set_xlabel(r"$g_2/2\pi$ (MHz)")
ax.set_ylabel(r"$\Lambda = \ln|\mu_{NS}|/T$ (ns$^{-1}$)")
ax.legend(fontsize=5, loc="center left", bbox_to_anchor=(1, 0.5))
plt.tight_layout()
plt.show()

# Rescaling

## 1. Conservative limit: collision point $g_{2,0}$ and slope $a$
At $\kappa_b = 0$ the period-2 orbit loses stability at $g_{2,0}$, where two pairs of multipliers collide on the unit circle (Hamiltonian–Hopf, Krein collision).
Above it, $\sigma_H^2 \simeq a\,(g_2 - g_{2,0})$, with $\sigma_H$ the largest real part of the exponents.
$g_{2,0}$ is found by bisection (orbit unstable when $\sigma_H > 10^{-8}$/ns; below the collision $\sigma_H$ is at machine precision).
$a$ is the linear coefficient of a quadratic fit of $\sigma_H^2$ on $[g_{2,0}, g_{2,0} + 3$ MHz$]$, i.e. the local slope at the collision.

In [ ]:
def max_exponent(z, eps_p, kappa_b):
    # solve the orbit at (eps_p, kappa_b) from the guess z; return (z, Floquet exponents in 1/ns)
    set_params(eps_p, ratio*g2_of(eps_p), kappa_b)
    z = hb.solve_robust(z)
    return z, floquet_exponents(hb, z)

def threshold(kappa_b, unstable=lambda lam: np.max(lam.real) > 0, step=0.0025, n_bisect=30):
    # step down in eps_p from eps_p_top until the orbit is stable, then bisect. Returns (g2* in MHz, orbit just above)
    z, e = continue_in_kappa(z_ref, kappa_b_ref, kappa_b), eps_p_top
    while True:
        z_n, lam = max_exponent(z, e - step, kappa_b)
        if not unstable(lam):
            break
        z, e = z_n, e - step
    e_u, e_s = e, e - step
    for _ in range(n_bisect):
        e_m = (e_u + e_s)/2
        z_m, lam = max_exponent(z, e_m, kappa_b)
        if unstable(lam):
            z, e_u = z_m, e_m
        else:
            e_s = e_m
    return MHz(g2_of((e_u + e_s)/2)), z, e_u

In [ ]:
g2_0, z_0, e_0 = threshold(0.0, unstable=lambda lam: np.max(lam.real) > 1e-8)

# sigma_H on a fine grid above the collision
g2_H, sigma_H = [], []
z = z_0
for e in np.linspace(e_0, e_0 + 0.03, 61):
    z, lam = max_exponent(z, e, 0.0)
    g2_H.append(MHz(g2_of(e))); sigma_H.append(np.max(lam.real))
g2_H, sigma_H = np.array(g2_H), np.array(sigma_H)
m = g2_H - g2_0 < 3
a = np.polyfit(g2_H[m] - g2_0, sigma_H[m]**2, 2)[1]
print(f"g2,0/2pi = {g2_0:.4f} MHz, a = {a:.4f} /ns^2/MHz")

## 2. Threshold for $\kappa_b \to 0$
Thresholds by bisection (max $|\mu| > 1$), at small $\kappa_b$ and at the four values of the sweep.
The interpolated $g_2^*$ of the cell above is only accurate to about 0.05 MHz (coarse grid, steep $\Lambda$); these are used from now on.

In [ ]:
kappa_b_small = 2*np.pi*1e-3*np.array([0.25, 1, 2, 5, 10, 20])
g2_star_small = np.array([threshold(kap)[0] for kap in kappa_b_small])
g2_star_bis = np.array([threshold(kap)[0] for kap in kappa_b_list])

kap_all = np.concatenate([kappa_b_small, kappa_b_list])
g2_all = np.concatenate([g2_star_small, g2_star_bis])
for kap, g in zip(kap_all, g2_all):
    print(f"kappa_b/2pi = {MHz(kap):6.2f} MHz: g2*/2pi = {g:.4f} MHz, g2* - g2,0 = {g - g2_0:+.4f} MHz")

# g2* = g2*(0+) + c kappa_b^2, fitted on kappa_b/2pi <= 48 MHz
m = kap_all <= 2*np.pi*0.048 + 1e-12
c2, g2_star_0 = np.polyfit(kap_all[m]**2, g2_all[m], 1)
print(f"\ng2*(kappa_b -> 0)/2pi = {g2_star_0:.4f} MHz, i.e. {g2_star_0 - g2_0:+.4f} MHz from the conservative collision")
print(f"g2* - g2*(0+) = {c2:.3f} MHz x (kappa_b [1/ns])^2")

**The vanishing-loss limit of the threshold is not the conservative collision.** As $\kappa_b\to0$, $g_2^*$ tends to a value about 0.15 MHz *below* $g_{2,0}$
(converged in $N_H$: 15 and 20 harmonics agree to $10^{-3}$ MHz), and rises from there as $\kappa_b^2$.
An arbitrarily small loss destabilises the orbit slightly *before* the Krein collision. This is the dissipation-induced instability of Hamiltonian–Hopf points
(Ziegler's paradox; Bottema 1956, Kirillov, *Nonconservative Stability Problems*, 2013).

**Two-mode model.** Near the collision, only the two colliding modes matter. Their mean damping is fixed by Liouville: the two pairs are all four multipliers, so
$\ln|\mu_1| + \ln|\mu_2| = -\kappa_b T/2$, a mean rate of $-\kappa_b/4$. The generic unfolding is
$$\Lambda = -\frac{\kappa_b}{4} + \mathrm{Re}\sqrt{a(g_2 - g_{2,0}) + iG\kappa_b + e\kappa_b^2}.$$
$G$ comes from the damping being different on the two modes (only the buffer is damped), $e$ from second-order shifts. Setting $\Lambda = 0$ gives
$$a\,(g_2^* - g_{2,0}) = -4G^2 + \left(\tfrac{1}{16} - e\right)\kappa_b^2,$$
i.e. a finite offset at $\kappa_b\to0$ and a quadratic rise: both seen above. $G$ and $e$ are fitted on the thresholds; the shape of $\Lambda$ is then a prediction.

**Rescaling.** With $x = a(g_2 - g_{2,0})/\kappa_b^2$,
$$\frac{\Lambda}{\kappa_b} = -\frac14 + \mathrm{Re}\sqrt{x + e + iG/\kappa_b}.$$
The curves collapse when $G/\kappa_b \ll 1$ ($G/\kappa_b = 0.30$ at 48 MHz down to 0.04 at 384 MHz). They then follow $-1/4 + \sqrt{x + e}$, which tends to $\sqrt{x} - 1/4$ at large $x$.
Measuring $g_2$ from $g_2^*(0^+)$ instead of $g_{2,0}$ moves $x$ by $4G^2/\kappa_b^2$ ($a\,(g_2^*(0^+) - g_{2,0}) = -4G^2$). With $x' = a(g_2 - g_2^*(0^+))/\kappa_b^2$ and $\gamma = G/\kappa_b$,
$$\frac{\Lambda}{\kappa_b} = -\frac14 + \mathrm{Re}\sqrt{x' + e - 4\gamma^2 + i\gamma},$$
whose zero is at $x'^* = 1/16 - e$ **for every** $\kappa_b$: in this variable the threshold collapses exactly (within the model), and only the shape near it still depends on $\gamma$.

In [ ]:
G = np.sqrt((g2_0 - g2_star_0)*a/4)
e_fit = 1/16 - a*c2
print(f"G = {G:.4f} /ns^2, e = {e_fit:.3f};  G/kappa_b = " + ", ".join(f"{G/kap:.3f}" for kap in kappa_b_list))

def Lambda_model(g2, kappa_b):
    return -kappa_b/4 + np.real(np.sqrt(a*(g2 - g2_0) + 1j*G*kappa_b + e_fit*kappa_b**2 + 0j))

for k, kappa_b in enumerate(kappa_b_list):
    x_star = a*(g2_star_bis[k] - g2_0)/kappa_b**2
    x_star_0 = a*(g2_star_bis[k] - g2_star_0)/kappa_b**2
    print(f"kappa_b/2pi = {MHz(kappa_b):3.0f} MHz: x* = {x_star:.3f} (origin g2,0), {x_star_0:.3f} (origin g2*(0+))")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(2*width_in, height_in), sharey=True)
x_th = np.linspace(-e_fit, 6, 300)
for ax, origin, name in ((axes[0], g2_0, r"g_{2,0}"), (axes[1], g2_star_0, r"g_2^*(0^+)")):
    for k, kappa_b in enumerate(kappa_b_list):
        ax.plot(a*(g2_MHz[k] - origin)/kappa_b**2, Lambda[k]/kappa_b, color=colors[k], label=rf"$\kappa_b/2\pi = {MHz(kappa_b):.0f}$ MHz")
    ax.plot(x_th, np.sqrt(np.clip(x_th + e_fit, 0, None)) - 1/4, "--", color="grey", lw=0.8, label=rf"$\sqrt{{x + e}} - 1/4$, $e = {e_fit:.2f}$")
    ax.axhline(0, color="grey", lw=0.5)
    ax.set_xlim(-1, 6)
    ax.set_ylim(-0.6, 2.5)
    ax.grid(False)
    ax.set_xlabel(rf"$x = a\,(g_2 - {name})/\kappa_b^2$")
axes[0].set_ylabel(r"$\Lambda/\kappa_b$")
axes[1].legend(fontsize=5, loc="center left", bbox_to_anchor=(1, 0.5))
plt.tight_layout()
plt.show()

**Reading the collapse.**
- Left (origin $g_{2,0}$): the curves meet at large $x$, but the zero crossings spread ($x^*$ from $-0.03$ at 48 MHz to 0.40 at 384 MHz, most of it below 120 MHz). That spread is the offset $-4G^2/\kappa_b^2$, which grows as $\kappa_b$ decreases.
- Right (origin $g_2^*(0^+)$): the offset is absorbed and the zero crossings collapse ($x^* = 0.32$–$0.40$, rising slowly with $\kappa_b$). The remaining drift at 384 MHz is the departure from the pure $\kappa_b^2$ law (local exponent 2.2 there).
- The grey curve is the model for $G/\kappa_b\to0$, $\sqrt{x + e} - 1/4$, with $e$ fitted on the thresholds (no new parameter). The shift $-e\simeq0.26$ in $x$ is the real $\kappa_b^2$ term of $D$.
  Its zero is at $x^* = 1/16 - e = 0.32$. The data leave it at large $\kappa_b$ ($x^* = 0.40$ at 384 MHz: $e$ is not constant there, Sec. 5),
  and sit slightly below it at large $x$ because $\sigma_H^2$ bends down away from the collision ($a$ is its slope at $g_{2,0}$).

Check of the two-mode model below: $\Lambda$ against $g_2$ (solid, Hill) and the model with $G$, $e$ fitted on the thresholds only (dotted).

In [ ]:
fig, ax = plt.subplots()
for k, kappa_b in enumerate(kappa_b_list):
    ax.plot(g2_MHz[k], Lambda[k], color=colors[k], label=rf"$\kappa_b/2\pi = {MHz(kappa_b):.0f}$ MHz")
    ax.plot(g2_MHz[k], Lambda_model(g2_MHz[k], kappa_b), ls=":", color="k", lw=0.8)
ax.plot([], [], ls=":", color="k", lw=0.8, label="two-mode model")
ax.axhline(0, color="grey", lw=0.5)
ax.set_xlim(74, 95)
ax.set_ylim(-0.6, 1.5)
ax.grid(False)
ax.set_xlabel(r"$g_2/2\pi$ (MHz)")
ax.set_ylabel(r"$\Lambda$ (ns$^{-1}$)")
ax.legend(fontsize=5, loc="center left", bbox_to_anchor=(1, 0.5))
plt.tight_layout()
plt.show()

**Model check.** $G$ and $e$ are fitted only on the thresholds with $\kappa_b/2\pi \le 48$ MHz. With them, the model reproduces the shape of $\Lambda$ around the threshold up to about 120 MHz,
including the slow rise below it, and the threshold to within 0.2 MHz up to about 200 MHz. At 384 MHz it puts the threshold too low, at 85.1 MHz instead of 87.4 MHz.
There the threshold rises faster than $\kappa_b^2$ (local exponent 2.2), so $e$ is not constant.
At large $g_2$ the model lies above the data for the same reason as the asymptote: $a$ is the slope of $\sigma_H^2$ at the collision.

# Check of the two-mode model from the Floquet spectrum

## 3. The discriminant $D$, computed directly
Keep one exponent per complex pair: the two exponents with $\mathrm{Im}\,\lambda > 0$, $\lambda_1$ and $\lambda_2$. They are the eigenvalues of the reduced $2\times2$ generator $R$, so
$$\frac{\lambda_1 + \lambda_2}{2} = \frac{\mathrm{tr}\,R}{2},\qquad D = \left(\frac{\lambda_1 - \lambda_2}{2}\right)^2,\qquad \Lambda = \mathrm{Re}\,\frac{\mathrm{tr}\,R}{2} + \mathrm{Re}\sqrt D.$$
This uses no model: $D$ is a symmetric function of the two exponents, so it is analytic in $(g_2, \kappa_b)$ even where $\lambda_{1,2}$ themselves have a square-root branch point (the collision).
The reduction fails only where one pair reaches the real axis: there, only one exponent is left with $\mathrm{Im}\,\lambda > 0$ ($D$ = NaN below).

**Two exact properties.**
- **Liouville:** $\mathrm{Re}\,\mathrm{tr}R/2 = -\kappa_b/4$.
- **Time reversal:** $\tau\to-\tau$ maps the equations at $\kappa_b$ onto those at $-\kappa_b$ (the drive $\cos4\tau$ is even). The orbit at $-\kappa_b$ is then the time-reversed orbit, and the exponents change sign: $\lambda(-\kappa_b) = -\lambda(\kappa_b)$.
  For the upper-half-plane members this gives $D(g_2, -\kappa_b) = D(g_2, \kappa_b)^*$.
  So in the expansion $D = \sum d_{mn}(g_2 - g_{2,0})^m\kappa_b^n$, **$d_{mn}$ is real for even $n$ and imaginary for odd $n$, to all orders**, including the deformation of the orbit with $\kappa_b$.
  This is the reason for the $i$ in front of $G$; the count of factors of $i$ in the $2\times2$ matrix is the first-order version of it.

In [ ]:
def eps_of_MHz(g):
    return np.arcsin(g*2*np.pi*1e-3/(E_J*phi_a**2*phi_b))

def D_and_mean(lam):
    # the two upper-half-plane exponents (one per pair): D = ((l1 - l2)/2)^2, mean = (l1 + l2)/2. NaN when a pair is real.
    up = lam[lam.imag > 1e-9]
    if len(up) != 2:
        return np.nan, np.nan
    return ((up[0] - up[1])/2)**2, (up[0] + up[1])/2

def D_grid(delta, kappas):
    # D and mean exponent at g2 = g2,0 + delta (MHz), for each kappa_b. Each row is continued down from eps_p_top.
    # status: 0 = both pairs complex, 1 = one pair real (reduction undefined), 2 = no period-2 orbit (period doubling or solver failure)
    D = np.full((len(kappas), len(delta)), np.nan, complex)
    mean = D.copy()
    status = np.full(D.shape, 2)
    e_targets = eps_of_MHz(g2_0 + np.asarray(delta))[::-1]      # descending eps_p
    for i, kap in enumerate(kappas):
        z, e = continue_in_kappa(z_ref, kappa_b_ref, kap), eps_p_top
        while e - 0.0025 > e_targets[0]:
            e -= 0.0025
            z, _ = max_exponent(z, e, kap)
        for j, et in enumerate(e_targets):
            try:
                z, lam = max_exponent(z, et, kap)
            except RuntimeError:
                break
            if odd_content(z, N_H) < odd_min:
                break
            jj = len(delta) - 1 - j
            D[i, jj], mean[i, jj] = D_and_mean(lam)
            status[i, jj] = 0 if np.isfinite(D[i, jj]) else 1
    return D, mean, status

In [ ]:
# local grid around the collision: used for the checks and the Taylor models
delta_loc = np.linspace(-0.75, 1.0, 29)
kap_loc = 2*np.pi*1e-3*np.linspace(0, 100, 21)
D_loc, mean_loc, status_loc = D_grid(delta_loc, kap_loc)
print(f"{np.sum(status_loc == 0)} / {status_loc.size} points with both pairs complex")
print(f"Liouville: max |Re (l1 + l2)/2 + kappa_b/4| = {np.nanmax(np.abs(mean_loc.real + kap_loc[:, None]/4)):.1e} /ns")
print(f"Time reversal at kappa_b = 0: max |Im D| = {np.nanmax(np.abs(D_loc[0].imag)):.1e} /ns^2")

## 4. First-principles check of $G$ and of the offset
**$G$ at the collision.** $G = \mathrm{Im}\,\partial_{\kappa_b} D$ at $(g_{2,0}, 0)$, from a fit of $D(g_{2,0}, \kappa_b)$ in powers of $\kappa_b$ ($\kappa_b/2\pi \le 100$ MHz).
The same fit gives $e = \mathrm{Re}\,\tfrac12\partial^2_{\kappa_b}D$ and checks the parity rule: the real part of the odd coefficients and the imaginary part of the even ones should vanish.

**Offset, exact as $\kappa_b\to0$.** To first order in $\kappa_b$, $D = D_0(g_2) + iG(g_2)\kappa_b$, with $D_0 < 0$ just below the collision. Then $\mathrm{Re}\sqrt D = G\kappa_b/2\sqrt{|D_0|}$, and $\Lambda = 0$ gives
$$D_0(g_2^*) + 4G(g_2^*)^2 = 0\qquad(\kappa_b\to0).$$
This uses only the lossless spectrum, $D_0$, and its first-order change with loss, $G(g_2) = \mathrm{Im}\,D(g_2,\kappa_1)/\kappa_1$ (here $\kappa_1/2\pi = 1$ MHz). No fit to the thresholds is involved.

In [ ]:
# G and e at the collision
j0 = np.argmin(np.abs(delta_loc))
for deg in (3, 4, 5):
    c_re = np.polyfit(kap_loc, D_loc[:, j0].real, deg)[::-1]
    c_im = np.polyfit(kap_loc, D_loc[:, j0].imag, deg)[::-1]
    print(f"degree {deg}: Re d_0n = {np.round(c_re, 4)},  Im d_0n = {np.round(c_im, 4)}")
G_direct, e_direct = c_im[1], c_re[2]
print(f"\nG = {G_direct:.4f} /ns^2 (direct), {G:.4f} (fitted on the thresholds);  e = {e_direct:.3f} (direct), {e_fit:.3f} (fitted)")

# offset for kappa_b -> 0
delta_f = np.linspace(-0.4, 0.1, 51)
kappa_1 = 2*np.pi*1e-3
D_f, _, _ = D_grid(delta_f, np.array([0.0, kappa_1]))
D0_f, G_f = D_f[0].real, D_f[1].imag/kappa_1
F = D0_f + 4*G_f**2
i = np.where(np.diff(np.sign(F)) != 0)[0][0]
delta_star_pred = np.interp(0, F[i:i + 2], delta_f[i:i + 2])
print(f"\ng2*(0+) - g2,0 = {delta_star_pred:+.4f} MHz predicted from D0 + 4G^2 = 0;  {g2_star_0 - g2_0:+.4f} MHz from the Hill thresholds")
print(f"(G there = {np.interp(delta_star_pred, delta_f, G_f):.4f} /ns^2, D0 there = {np.interp(delta_star_pred, delta_f, D0_f):.4f} /ns^2)")

## 5. Higher orders
Truncated model of order $N$: all monomials $(g_2 - g_{2,0})^m\kappa_b^n$ with $2m + n \le N$, with the parity rule above. The weight 2 for $g_2 - g_{2,0}$ is the power counting at the threshold,
where $g_2^* - g_{2,0}\propto\kappa_b^2$. $N = 2$ is the model of Sec. 2 ($a$, $G$, $e$) plus a constant. The coefficients are fitted on $D$ over the local grid, not on the thresholds.
Each model gives $\Lambda = -\kappa_b/4 + \mathrm{Re}\sqrt{D_N}$, and its threshold is compared with the Hill bisection.

**Will higher orders always do better?** Only inside the domain where the expansion of $D$ converges. That domain is limited by the nearest point where $D$ is not analytic:
where one of the two pairs reaches the real axis. At $\kappa_b = 0$ that is the period doubling where the period-2 orbit is born (multiplier $+1$), 1.55 MHz below $g_{2,0}$.
With loss, this line moves up in $g_2$ (map below), and at large $\kappa_b$ it comes close to the threshold. Beyond it, adding orders makes the prediction worse, as for any Taylor series outside its radius of convergence.

In [ ]:
def monomials(N):
    return [(m, k) for m in range(N//2 + 1) for k in range(N + 1) if 2*m + k <= N]

def fit_D(N, D, delta, kappas):
    # least squares with the parity rule: real coefficients for even powers of kappa_b, imaginary for odd ones
    dd, kk = np.meshgrid(delta, kappas)
    ok = np.isfinite(D)
    dd, kk, y = dd[ok], kk[ok], D[ok]
    terms_re = [t for t in monomials(N) if t[1] % 2 == 0]
    terms_im = [t for t in monomials(N) if t[1] % 2 == 1]
    c_re = np.linalg.lstsq(np.array([dd**m*kk**k for m, k in terms_re]).T, y.real, rcond=None)[0]
    c_im = np.linalg.lstsq(np.array([dd**m*kk**k for m, k in terms_im]).T, y.imag, rcond=None)[0]
    def D_N(dl, kap):
        return (sum(c*dl**m*kap**k for c, (m, k) in zip(c_re, terms_re))
                + 1j*sum(c*dl**m*kap**k for c, (m, k) in zip(c_im, terms_im)))
    return D_N, np.max(np.abs(D_N(dd, kk) - y))

def threshold_model(D_N, kap, dl=np.linspace(-1.5, 15, 33001)):
    L = -kap/4 + np.real(np.sqrt(D_N(dl, kap) + 0j))
    i = np.where((L[:-1] <= 0) & (L[1:] > 0))[0]
    return dl[i[0]] if len(i) else np.nan

orders = [2, 3, 4, 5, 6]
dg_hill = g2_all - g2_0
dg_model = {}
print("kappa_b/2pi (MHz):      " + "".join(f"{MHz(k):9.2f}" for k in kap_all))
print("Hill g2* - g2,0 (MHz):  " + "".join(f"{x:9.4f}" for x in dg_hill))
for N in orders:
    D_N, res = fit_D(N, D_loc, delta_loc, kap_loc)
    dg_model[N] = np.array([threshold_model(D_N, k) for k in kap_all])
    print(f"N = {N} (residual {res:.0e}):  " + "".join(f"{x:9.4f}" for x in dg_model[N]))

In [ ]:
# where the reduction is defined: wider grid, with the thresholds
delta_map = np.arange(-1.5, 12.01, 0.25)
kap_map = 2*np.pi*1e-3*np.arange(0, 401, 25)
_, _, status_map = D_grid(delta_map, kap_map)

In [ ]:
from matplotlib.colors import ListedColormap
fig, axes = plt.subplots(1, 2, figsize=(2*width_in, height_in))

ax = axes[0]
for k_sel in [np.argmin(np.abs(MHz(kap_all) - f)) for f in (0.25, 20, 48, 96, 192)]:
    err = [abs(dg_model[N][k_sel] - dg_hill[k_sel]) for N in orders]
    ax.semilogy(orders, err, "o-", ms=3, label=rf"$\kappa_b/2\pi = {MHz(kap_all[k_sel]):.3g}$ MHz")
ax.set_xticks(orders)
ax.set_xlabel(r"order $N$")
ax.set_ylabel(r"$|g_2^*{}^{\rm model} - g_2^*{}^{\rm Hill}|/2\pi$ (MHz)")
ax.grid(False)
ax.legend(fontsize=6)

ax = axes[1]
ax.pcolormesh(MHz(kap_map), g2_0 + delta_map, status_map.T, shading="nearest",
              cmap=ListedColormap(["white", "tab:orange", "lightgrey"]), vmin=0, vmax=2)
ax.plot(MHz(kap_all), g2_all, "k.-", ms=3, label=r"threshold $g_2^*$")
ax.add_patch(plt.Rectangle((0, g2_0 + delta_loc[0]), MHz(kap_loc[-1]), delta_loc[-1] - delta_loc[0], fill=False, ls="--", lw=0.8))
ax.plot([], [], "s", color="tab:orange", label="one pair real")
ax.plot([], [], "s", color="lightgrey", label="no period-2 orbit")
ax.plot([], [], "k--", lw=0.8, label="fit domain")
ax.set_xlim(0, 400)
ax.set_ylim(g2_0 + delta_map[0], g2_0 + delta_map[-1])
ax.set_xlabel(r"$\kappa_b/2\pi$ (MHz)")
ax.set_ylabel(r"$g_2/2\pi$ (MHz)")
ax.grid(False)
ax.legend(fontsize=6, loc="upper left")
plt.tight_layout()
plt.show()

**Results.**
- **Exact properties.** Liouville holds to $10^{-4}$/ns, $\mathrm{Im}\,D = 0$ at $\kappa_b = 0$ to $10^{-13}$, and the parity rule holds: as the degree of the fit in $\kappa_b$ increases,
  the real part of the $\kappa_b^1$ coefficient and the imaginary part of the $\kappa_b^0$, $\kappa_b^2$ coefficients go to zero.
- **$G$ from the spectrum:** 0.096 /ns$^2$ at $g_{2,0}$, against 0.090 fitted on the thresholds. The difference is not an error of the model: $G$ depends on $g_2$ (0.0945 at $g_2^*(0^+)$),
  and $D_0$ is not linear ($D_0(g_2^*) = -0.0357$ against $a\,(g_2^* - g_{2,0}) = -0.0325$). The order-2 fit on the thresholds absorbs both into an effective $G$.
- **The offset is predicted, not fitted.** $D_0 + 4G^2 = 0$, with $D_0$ and $G$ taken from the spectrum, gives $g_2^*(0^+) - g_{2,0} = -0.1546$ MHz, against $-0.1545$ MHz from the Hill thresholds.
- **$e$** is poorly determined ($-0.22$ to $-0.33$ depending on the degree of the fit): the $\kappa_b$ series of $D$ converges slowly.
- **Higher orders.** For $\kappa_b/2\pi \le 48$ MHz the threshold error falls from about $3\times10^{-2}$ MHz ($N = 2$) to $2$–$4\times10^{-3}$ MHz ($N = 6$), slowly and not monotonically.
  The odd orders add only imaginary terms ($\delta\kappa_b$, $\kappa_b^3$), which barely move the threshold. At 85 MHz, near the edge of the fit domain, the error drops at $N = 4$ and then rises again.
  At 197 MHz it gets worse from $N = 4$ on, and at 384 MHz the models fail. These thresholds are 2.4 and 11 MHz above $g_{2,0}$, while the expansion around $(g_{2,0}, 0)$ cannot reach farther than
  the line where a pair becomes real: the period doubling, 1.55 MHz below $g_{2,0}$ at $\kappa_b = 0$ (orange and grey in the map).
- So $\Lambda = -\kappa_b/4 + \mathrm{Re}\sqrt D$ is exact, and all the approximation is in the representation of $D$. A low-order expansion around the collision explains the offset and the
  $\kappa_b^2$ law at small $\kappa_b$. It cannot be pushed to large $\kappa_b$ by adding orders: that would need an expansion around a point closer to the threshold of interest.